# Table III: Wildfire ignition model comparison

This notebook extends `replication.ipynb` with a model-comparison workflow for the wildfire ignition task. Classical baselines flatten each 75-day sequence into `15 variables x 75 timesteps = 1125` features, while the LSTM-style models use the 3D sequence tensor.

In [ ]:
import random
import warnings

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

from sklearn.base import clone
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.utils.class_weight import compute_class_weight

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

try:
    from xgboost import XGBClassifier
    XGBOOST_AVAILABLE = True
except ImportError:
    XGBClassifier = None
    XGBOOST_AVAILABLE = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

## Data preparation

This follows the style of `replication.ipynb`: remove GRIDMET fill-value rows, sort by location/date, build 75-day sequences, and label each sequence as positive if any day in that block has `Wildfire == "Yes"`.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    seqs,
    labels,
    test_size=0.20,
    stratify=labels,
    random_state=SEED,
)

print("Train:", X_train.shape, "Test:", X_test.shape)
print("Train positives:", y_train.sum(), "/", len(y_train))
print("Test positives:", y_test.sum(), "/", len(y_test))

## Shared evaluation helpers

In [ ]:
def flatten_sequences(X):
    return X.reshape(X.shape[0], -1)


def metric_row(model_name, y_true, y_pred):
    return {
        "Model": model_name,
        "Accuracy [%]": 100 * accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall": recall_score(y_true, y_pred, zero_division=0),
        "F1": f1_score(y_true, y_pred, zero_division=0),
    }


def display_table(rows):
    table = pd.DataFrame(rows)
    table = table.sort_values("Accuracy [%]", ascending=False).reset_index(drop=True)
    return table.round({"Accuracy [%]": 1, "Precision": 2, "Recall": 2, "F1": 2})


results = []

## Classical baselines

These models are not inherently time-series-aware, so the 3D sequence is flattened to 1125 features before scaling/modeling.

In [ ]:
flat_scaled_pipeline = lambda estimator: Pipeline([
    ("flatten", FunctionTransformer(flatten_sequences, validate=False)),
    ("scaler", StandardScaler()),
    ("model", estimator),
])

flat_unscaled_pipeline = lambda estimator: Pipeline([
    ("flatten", FunctionTransformer(flatten_sequences, validate=False)),
    ("model", estimator),
])

classical_models = [
    ("Gradient Boosting*", flat_unscaled_pipeline(GradientBoostingClassifier(random_state=SEED))),
    ("Random Forest*", flat_unscaled_pipeline(RandomForestClassifier(n_estimators=300, class_weight="balanced", n_jobs=-1, random_state=SEED))),
    ("K-Nearest Neighbors*", flat_scaled_pipeline(KNeighborsClassifier(n_neighbors=5))),
    ("Decision Tree*", flat_unscaled_pipeline(DecisionTreeClassifier(class_weight="balanced", random_state=SEED))),
    ("Logistic Regression*", flat_scaled_pipeline(LogisticRegression(max_iter=2000, class_weight="balanced", n_jobs=-1, random_state=SEED))),
    ("Naive Bayes*", flat_unscaled_pipeline(GaussianNB())),
]

if XGBOOST_AVAILABLE:
    classical_models.insert(
        0,
        (
            "XGBoost*",
            flat_unscaled_pipeline(
                XGBClassifier(
                    n_estimators=300,
                    max_depth=4,
                    learning_rate=0.05,
                    subsample=0.9,
                    colsample_bytree=0.9,
                    eval_metric="logloss",
                    random_state=SEED,
                )
            ),
        ),
    )
else:
    print("XGBoost is not installed, so the XGBoost row will be skipped. Install with: pip install xgboost")

for name, estimator in classical_models:
    print(f"Fitting {name}...")
    estimator.fit(X_train, y_train)
    y_pred = estimator.predict(X_test)
    results.append(metric_row(name, y_test, y_pred))

display_table(results)

## Simple MLP baseline

The MLP baseline also uses flattened 1125-dimensional feature vectors.

In [ ]:
class MLPClassifierTorch(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.30),
            nn.Linear(256, 64),
            nn.ReLU(),
            nn.Dropout(0.20),
            nn.Linear(64, 2),
        )

    def forward(self, x):
        return self.net(x)


def make_loader(X, y, batch_size=128, shuffle=False):
    dataset = TensorDataset(torch.tensor(X, dtype=torch.float32), torch.tensor(y, dtype=torch.long))
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle)


def train_torch_classifier(model, train_loader, epochs=8, lr=1e-3):
    model = model.to(device)
    class_weights = compute_class_weight(class_weight="balanced", classes=np.array([0, 1]), y=y_train)
    criterion = nn.CrossEntropyLoss(weight=torch.tensor(class_weights, dtype=torch.float32, device=device))
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    for epoch in range(epochs):
        model.train()
        total_loss = 0.0
        for Xb, yb in train_loader:
            Xb, yb = Xb.to(device), yb.to(device)
            optimizer.zero_grad()
            loss = criterion(model(Xb), yb)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        print(f"Epoch {epoch + 1:02d}/{epochs}, loss={total_loss / len(train_loader):.4f}")
    return model


def predict_torch_classifier(model, X, batch_size=256):
    loader = make_loader(X, np.zeros(len(X), dtype=np.int64), batch_size=batch_size, shuffle=False)
    preds = []
    model.eval()
    with torch.no_grad():
        for Xb, _ in loader:
            logits = model(Xb.to(device))
            preds.extend(torch.argmax(logits, dim=1).cpu().numpy())
    return np.array(preds)


flat_scaler = StandardScaler()
X_train_flat_scaled = flat_scaler.fit_transform(flatten_sequences(X_train))
X_test_flat_scaled = flat_scaler.transform(flatten_sequences(X_test))

mlp = MLPClassifierTorch(input_dim=X_train_flat_scaled.shape[1])
mlp_train_loader = make_loader(X_train_flat_scaled, y_train, batch_size=128, shuffle=True)
mlp = train_torch_classifier(mlp, mlp_train_loader, epochs=8, lr=1e-3)
y_pred_mlp = predict_torch_classifier(mlp, X_test_flat_scaled)
results.append(metric_row("Simple-MLP*", y_test, y_pred_mlp))

display_table(results)

## Time-series-aware neural models

In [ ]:
seq_scaler = StandardScaler()
X_train_seq_scaled = seq_scaler.fit_transform(X_train.reshape(-1, len(FEATURES))).reshape(X_train.shape)
X_test_seq_scaled = seq_scaler.transform(X_test.reshape(-1, len(FEATURES))).reshape(X_test.shape)

seq_train_loader = make_loader(X_train_seq_scaled, y_train, batch_size=128, shuffle=True)


class TwoLayerLSTM(nn.Module):
    def __init__(self, input_dim, hidden_dim=64):
        super().__init__()
        self.lstm = nn.LSTM(input_dim, hidden_dim, num_layers=2, batch_first=True, dropout=0.30)
        self.fc = nn.Linear(hidden_dim, 2)

    def forward(self, x):
        _, (hn, _) = self.lstm(x)
        return self.fc(hn[-1])


class LightTSInspired(nn.Module):
    def __init__(self, input_dim, hidden_dim=64):
        super().__init__()
        self.temporal = nn.Sequential(
            nn.Conv1d(input_dim, hidden_dim, kernel_size=5, padding=2),
            nn.ReLU(),
            nn.Dropout(0.20),
            nn.Conv1d(hidden_dim, hidden_dim, kernel_size=3, padding=1),
            nn.ReLU(),
        )
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.fc = nn.Linear(hidden_dim, 2)

    def forward(self, x):
        x = x.transpose(1, 2)
        x = self.temporal(x)
        x = self.pool(x).squeeze(-1)
        return self.fc(x)


class CNNLSTM(nn.Module):
    def __init__(self, input_dim, conv_dim=64, hidden_dim=64):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv1d(input_dim, conv_dim, kernel_size=5, padding=2),
            nn.ReLU(),
            nn.BatchNorm1d(conv_dim),
            nn.Dropout(0.20),
        )
        self.lstm = nn.LSTM(conv_dim, hidden_dim, num_layers=1, batch_first=True)
        self.fc = nn.Sequential(nn.Dropout(0.30), nn.Linear(hidden_dim, 2))

    def forward(self, x):
        x = x.transpose(1, 2)
        x = self.conv(x)
        x = x.transpose(1, 2)
        _, (hn, _) = self.lstm(x)
        return self.fc(hn[-1])

In [ ]:
neural_models = [
    ("Two-Layer-LSTM", TwoLayerLSTM(input_dim=len(FEATURES)), 8, 1e-3),
    ("LightTS-Inspired", LightTSInspired(input_dim=len(FEATURES)), 8, 1e-3),
    ("CNN-LSTM (ours)", CNNLSTM(input_dim=len(FEATURES)), 10, 1e-3),
]

for name, torch_model, epochs, lr in neural_models:
    print(f"\nFitting {name}...")
    torch_model = train_torch_classifier(torch_model, seq_train_loader, epochs=epochs, lr=lr)
    y_pred = predict_torch_classifier(torch_model, X_test_seq_scaled)
    results.append(metric_row(name, y_test, y_pred))

table_iii = display_table(results)
table_iii

## Final table and optional export

In [ ]:
target_order = [
    "CNN-LSTM (ours)",
    "XGBoost*",
    "Gradient Boosting*",
    "Random Forest*",
    "K-Nearest Neighbors*",
    "Simple-MLP*",
    "Decision Tree*",
    "Two-Layer-LSTM",
    "LightTS-Inspired",
    "Logistic Regression*",
    "Naive Bayes*",
]

table_iii = display_table(results)
table_iii["Model"] = pd.Categorical(table_iii["Model"], categories=target_order, ordered=True)
table_iii = table_iii.sort_values("Model").reset_index(drop=True)
table_iii["Model"] = table_iii["Model"].astype(str)

print("TABLE III: Wildfire ignition model comparison")
display(table_iii)
print("* Baselines not inherently time-series-aware; feature vectors flattened to 1125 dimensions (15 variables x 75 timesteps).")

table_iii.to_csv("table_iii_model_comparison.csv", index=False)